## 3-5 · Pandas와 DuckDB를 이용한 하이브리드 가공

In [1]:
! ls -l make_orders.py orders.csv

ls: make_orders.py: No such file or directory
-rw-r--r--@ 1 hyunjunson  staff  36244 10월  9 14:32 orders.csv


In [2]:
! uv run python -c "import sys, pandas, pyarrow, duckdb; print(pandas.__version__, pyarrow.__version__, duckdb.__version__, sys.prefix)"

3.0.6 25.0.1 1.5.6 /Users/hyunjunson/study/data-eng/week3/.venv


In [3]:
import duckdb
 
baseline = duckdb.sql("""
    SELECT count(*)                                AS rows,
           count(*) - count(price)                 AS null_price,
           count(*) - count(DISTINCT order_id)     AS dup_orders,
           count(DISTINCT category)                AS category_values
    FROM 'orders.csv'
""").df()

baseline

,rows,null_price,dup_orders,category_values
0,1015,30,15,5


In [4]:
# 1단계(오염 확인) — baseline.py 끝에 이어 붙입니다
# 어떤 값이 오염됐는지 눈으로 확인
print(duckdb.sql("""
    SELECT category, count(*) AS cnt
    FROM 'orders.csv'
    GROUP BY category
    ORDER BY cnt DESC
"""))

┌──────────┬───────┐
│ category │  cnt  │
│ varchar  │ int64 │
├──────────┼───────┤
│ 식품     │   396 │
│ 의류     │   284 │
│ 가전     │   203 │
│ 도서     │   112 │
│  식품    │    20 │
└──────────┴───────┘



In [ ]:
import duckdb
import pandas as pd
 
 
def load_raw() -> pd.DataFrame:
    # 추출 — 대용량이어도 여기서 필터·열 선택으로 줄일 수 있다
    return duckdb.sql("SELECT * FROM 'orders.csv'").df()
 
 
def clean(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['category'] = out['category'].str.strip()          # 공백 오염 제거
    out['order_date'] = pd.to_datetime(out['order_date'])  # 타입 복원
    out = out.drop_duplicates(subset='order_id', keep='first')  # 중복 제거
    out['price'] = out['price'].fillna(out['price'].median())   # 결측 보정
    out['amount'] = out['price'] * out['quantity']         # 파생 컬럼
    return out
 
 ç
if __name__ == '__main__':
    raw = load_raw()
    cleaned = clean(raw)
    cleaned.to_parquet('orders_clean.parquet', index=False)
    print(len(raw), '행 ->', len(cleaned), '행 정제 완료')

1015 행 -> 1000 행 정제 완료


In [6]:
# check_rerun.py — 3단계 단독 파일 (clean_orders.py 와 같은 폴더에 둡니다)
import pandas as pd
from clean_orders import load_raw, clean

first = clean(load_raw())
second = clean(load_raw())

print('행 수 같은가 :', len(first) == len(second))
print('내용 같은가 :', first.equals(second))

# 저장했다 읽어도 같은지
first.to_parquet('t1.parquet', index=False)
second.to_parquet('t2.parquet', index=False)
print('파일로도 같은가:', pd.read_parquet('t1.parquet').equals(pd.read_parquet('t2.parquet')))

ImportError: cannot import name 'load_raw' from 'clean_orders' (/Users/hyunjunson/study/data-eng/week3/clean_orders.py)